In [0]:
# Step 1 - Read Bronze Table
from pyspark.sql.functions import *

spark.sql("USE CATALOG Silver_Catalog")
spark.sql("USE SCHEMA Silver_SCH")

grid_df = spark.table("Bronze_Catalog.Bronze_SCH.Bronze_Grid_Load")

In [0]:
# Step 2 - Check Schema

grid_df.printSchema()

In [0]:
grid_df.select(
    "grid_voltage"
).show(10, False)

In [0]:
from pyspark.sql.functions import expr

grid_df = grid_df \
.withColumn("grid_voltage", expr("try_cast(grid_voltage as double)")) \
.withColumn("grid_current", expr("try_cast(grid_current as double)")) \
.withColumn("grid_load_kw", expr("try_cast(grid_load_kw as double)")) \
.withColumn("transformer_load", expr("try_cast(transformer_load as double)")) \
.withColumn("line_loss_percent", expr("try_cast(line_loss_percent as double)")) \
.withColumn("load_variation", expr("try_cast(load_variation as double)")) \
.withColumn("frequency_variation", expr("try_cast(frequency_variation as double)")) \
.withColumn("grid_capacity_kw", expr("try_cast(grid_capacity_kw as double)")) \
.withColumn("demand_forecast_kw", expr("try_cast(demand_forecast_kw as double)")) \
.withColumn("reserve_margin", expr("try_cast(reserve_margin as double)"))

In [0]:
grid_df = grid_df.filter(
    (col("grid_voltage") >= 0) &
    (col("grid_current") >= 0) &
    (col("grid_load_kw") >= 0) &
    (col("transformer_load") >= 0) &
    (col("line_loss_percent") >= 0) &
    (col("load_variation") >= 0) &
    (col("frequency_variation") >= 0) &
    (col("grid_capacity_kw") >= 0) &
    (col("demand_forecast_kw") >= 0) &
    (col("reserve_margin") >= 0)
)



In [0]:
from pyspark.sql.functions import col

grid_df = grid_df.filter(
    (col("grid_voltage").cast("float") >= 0) &
    (col("grid_current").cast("float") >= 0) &
    (col("grid_load_kw").cast("float") >= 0) &
    (col("transformer_load").cast("float") >= 0) &
    (col("line_loss_percent").cast("float") >= 0) &
    (col("load_variation").cast("float") >= 0) &
    (col("frequency_variation").cast("float") >= 0) &
    (col("grid_capacity_kw").cast("float") >= 0) &
    (col("demand_forecast_kw").cast("float") >= 0) &
    (col("reserve_margin").cast("float") >= 0)
)

print("Valid Records :", grid_df.count())

In [0]:
# Step 3 - Preview Data

grid_df.show(5, False)

In [0]:
# Step 5 - Check Null Values
from pyspark.sql.functions import col, sum

null_df = grid_df.select(
[
    sum(col(c).isNull().cast("int")).alias(c)
    for c in grid_df.columns
]
)

display(null_df)

In [0]:
# Step 6 - Convert Numeric Columns
from pyspark.sql.functions import expr

grid_df = grid_df \
.withColumn("grid_voltage", expr("try_cast(grid_voltage as double)")) \
.withColumn("grid_current", expr("try_cast(grid_current as double)")) \
.withColumn("grid_load_kw", expr("try_cast(grid_load_kw as double)")) \
.withColumn("transformer_load", expr("try_cast(transformer_load as double)")) \
.withColumn("line_loss_percent", expr("try_cast(line_loss_percent as double)")) \
.withColumn("load_variation", expr("try_cast(load_variation as double)")) \
.withColumn("frequency_variation", expr("try_cast(frequency_variation as double)")) \
.withColumn("grid_capacity_kw", expr("try_cast(grid_capacity_kw as double)")) \
.withColumn("demand_forecast_kw", expr("try_cast(demand_forecast_kw as double)")) \
.withColumn("reserve_margin", expr("try_cast(reserve_margin as double)"))

In [0]:
# Step 7 - Fill Nulls
grid_df = grid_df.fillna({
    "grid_voltage":0,
    "grid_current":0,
    "grid_load_kw":0,
    "transformer_load":0,
    "line_loss_percent":0,
    "load_variation":0,
    "frequency_variation":0,
    "grid_capacity_kw":0,
    "demand_forecast_kw":0,
    "reserve_margin":0
})

In [0]:
# Step 8 - Verify Schema
grid_df.printSchema()

In [0]:
# Step 9 - Data Validation
print("Valid Records :", grid_df.count())

In [0]:
# Step 10 - Create Surrogate Key
from pyspark.sql.functions import monotonically_increasing_id

grid_df = grid_df.withColumn(
    "grid_sk",
    monotonically_increasing_id()
)



In [0]:
# Step 11 - Implement SCD Type 2
from pyspark.sql.functions import current_timestamp, lit

grid_df = grid_df \
.withColumn("effective_from", current_timestamp()) \
.withColumn("effective_to", lit(None).cast("timestamp")) \
.withColumn("is_current", lit(True))



In [0]:
# Step 12 - Write Silver Table
grid_df.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema","true") \
.saveAsTable("Silver_Catalog.Silver_SCH.Silver_Grid_Load")

In [0]:
from pyspark.sql.functions import current_timestamp

records_loaded = grid_df.count()

try:

    spark.sql(f"""
    INSERT INTO Silver_Catalog.Silver_SCH.Audit_Log
    VALUES(
        'Silver_Grid_Load',
        'Grid Silver Pipeline',
        current_timestamp(),
        {records_loaded},
        'SUCCESS',
        NULL
    )
    """)

    print("✅ Silver_Grid_Load Loaded Successfully")

    

except Exception as e:

    error_message = str(e).replace("'", "''")

    spark.sql(f"""
    INSERT INTO Silver_Catalog.Silver_SCH.Audit_Log
    VALUES(
        'Silver_Grid_Load',
        'Grid Silver Pipeline',
        current_timestamp(),
        0,
        'FAILED',
        '{error_message}'
    )
    """)

    print(f"❌ Silver_Grid_Load Failed : {error_message}")

   

    raise

In [0]:
spark.sql("""
OPTIMIZE Silver_Catalog.Silver_SCH.Silver_Grid_Load
""")

print("✅ OPTIMIZE Completed")

In [0]:
spark.sql("""
OPTIMIZE Silver_Catalog.Silver_SCH.Silver_Grid_Load
ZORDER BY (household_id)
""")

print("✅ ZORDER Completed")

In [0]:
spark.sql("""
VACUUM Silver_Catalog.Silver_SCH.Silver_Grid_Load
RETAIN 168 HOURS
""")

print("✅ VACUUM Completed")

In [0]:
spark.sql("""
DESCRIBE HISTORY Silver_Catalog.Silver_SCH.Silver_Grid_Load
""").show(truncate=False)

In [0]:
display(grid_df)